In [12]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [13]:
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [14]:
import sys, os
print('python executable:', sys.executable)
print('notebook cwd:', os.getcwd())
fp = '0_DataPreparation/data/data_final_mit_LAG/train_data.csv'
print('relative path ->', fp)
print('abs path ->', os.path.abspath(fp))
print('exists:', os.path.exists(fp))
# falls vorhanden, zeige Dateigröße / Beispiele
if os.path.exists(fp):
    print('size (bytes):', os.path.getsize(fp))
    import pandas as pd
    print('first rows:')
    print(pd.read_csv(fp, nrows=3))
else:
    # liste Verzeichnisse zur Orientierung
    print('listing 0_DataPreparation/data:')
    try:
        print(sorted(os.listdir('0_DataPreparation/data'))[:50])
    except Exception as e:
        print('could not list dir:', e)

python executable: /workspaces/bakery-sales-prediction/.venv-1/bin/python
notebook cwd: /workspaces/bakery-sales-prediction
relative path -> 0_DataPreparation/data/data_final_mit_LAG/train_data.csv
abs path -> /workspaces/bakery-sales-prediction/0_DataPreparation/data/data_final_mit_LAG/train_data.csv
exists: True
size (bytes): 1385820
first rows:
        id       Datum      Umsatz  Temperatur  Windgeschwindigkeit  \
0  1307011  2013-07-01  148.828353     17.8375                 15.0   
1  1307013  2013-07-01  201.198426     17.8375                 15.0   
2  1307014  2013-07-01   65.890169     17.8375                 15.0   

   KielerWoche  Umsatz_lag_1  Umsatz_lag_7  Umsatz_rolling_7  WettercodeDunst  \
0          0.0           NaN           NaN               NaN                0   
1          0.0    148.828353           NaN               NaN                0   
2          0.0    201.198426           NaN               NaN                0   

   ...  BewoelkungWolkenKlar  FeiertagCh

In [15]:
import os
os.chdir('/workspaces/bakery-sales-prediction')
print('now cwd:', os.getcwd())

now cwd: /workspaces/bakery-sales-prediction


#### r^2 berechnen

rse = sum (errors^2)/sum((actual-mean(actual))^2)
r^2 = 1-rse


In [16]:
train_data = pd.read_csv('0_DataPreparation/data/data_final_mit_LAG/train_data.csv')
train_data.corr(numeric_only=True)['Umsatz'].sort_values(ascending=False)

Umsatz                            1.000000
WarengruppeRolls                  0.670450
Umsatz_rolling_7                  0.476264
WarengruppeCake                   0.238443
Temperatur                        0.216569
August                            0.173527
FeiertagSilvester                 0.139786
Juli                              0.134890
Umsatz_lag_7                      0.115825
WochentagSonntag                  0.109829
WochentagSamstag                  0.097841
BewoelkungWolkenKlar              0.065834
TagVorFeiertag                    0.064889
KielerWoche                       0.053631
FeiertagPfingstmontag             0.035675
Juni                              0.034507
WettercodeExtremwetter            0.023212
FeiertagTagDerDeutschenEinheit    0.020858
FeiertagOstermontag               0.019673
FeiertagChristiHimmelfahrt        0.017004
September                         0.012460
Windgeschwindigkeit               0.011454
Umsatz_lag_1                      0.011418
FeiertagErs

In [17]:
## mehrere Variablen
### Problem: Kaggle erwartet beim hochladen der Predictions genau 1830 Zeilen, wenn wir Zeilen im Test Data Bereich rausschneiden gibt es Probleme

# Trainingsdaten laden

train_data = pd.read_csv('0_DataPreparation/data/data_final_mit_LAG/train_data.csv')

# Modell trainieren ohne Windgeschwindigkeit und Fastzeit
mod = smf.ols('Umsatz ~ Temperatur + Windgeschwindigkeit + KielerWoche + WarengruppeBread + WarengruppeRolls + WarengruppeCroissants + WarengruppeConfectionery + WarengruppeCake + WarengruppeSeasonalBread + WochentagFreitag + WochentagSamstag + WochentagSonntag + Januar + Februar + Maerz + April + Mai + Juni + Juli + August + September + Oktober + November + Dezember + BewoelkungWolkenKlar + BewoelkungWolkenBewoelkt + BewoelkungWolkenBedeckt + FeiertagChristiHimmelfahrt + FeiertagErsterMai + FeiertagHeiligabend + FeiertagOstermontag + FeiertagPfingstmontag + FeiertagSilvester + FeiertagTagDerDeutschenEinheit',
               data=train_data).fit()
print(mod.summary())

# Test-/Vorhersagedaten laden (die 1830 Zeilen ohne Umsatz)
test_data = pd.read_csv('0_DataPreparation/data/data_final_mit_LAG/test_data.csv')  

# Vorhersagen machen
test_data['Umsatz_Vorhersage'] = mod.predict(test_data)

# Nur ID und Umsatz_Vorhersage speichern
predictions_output = test_data[['id', 'Umsatz_Vorhersage']]
predictions_output.to_csv('0_DataPreparation/data/data_final_mit_LAG/predictions_preencoded_imputated.csv', index=False)


# Erste Vorhersagen anzeigen
print("\nErste 10 Vorhersagen:")
print(predictions_output.head(10))

#print("Warengruppe Dummy-Spalten:", warengruppe_cols)
#print("Wetter Dummy-Spalten:", wetter_cols)
#print("Bewoelkung Dummy-Spalten:", bewoelkung_cols)


                            OLS Regression Results                            
Dep. Variable:                 Umsatz   R-squared:                       0.761
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                     767.2
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:54:10   Log-Likelihood:                -42662.
No. Observations:                7487   AIC:                         8.539e+04
Df Residuals:                    7455   BIC:                         8.561e+04
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [18]:
# MAPE berechnen
from sklearn.metrics import mean_absolute_percentage_error

# Letzten 20% der Trainingsdaten als Validierungsset
val_data = train_data.tail(int(len(train_data) * 0.2))

# Vorhersagen auf Validierungsset
val_predictions = mod.predict(val_data)

# Nur Zeilen ohne fehlende Umsatz-Werte
mask = val_data['Umsatz'].notna()
mape = mean_absolute_percentage_error(val_data['Umsatz'][mask], val_predictions[mask])

print(f"Baseline MAPE: {mape * 100:.2f}%")

Baseline MAPE: 33.74%
